# TP 3 — Prompter des LLM locaux sur Tiny Shakespeare

J'identifie le locuteur d'une réplique avec des LLM prêts à l'emploi, sans entraîner de paramètres. Je garde les données, le découpage et la mesure des TP1 et TP2, car seul le pipeline change. L'ingénierie passe des features vers le prompt, son évaluation et son coût, donc le prompt devient un objet versionné et testé.

Je compare aussi deux modèles à budget mémoire comparable, car choisir un modèle est un choix de déploiement autant qu'un choix de qualité.

| Étape | Section | Ce que je livre |
|---|---|---|
| 1. Prompting | section 1 | Les 962 répliques, un jeu de mise au point, les repères ML, un prompt en blocs et une échelle d'itérations. |
| 2. Modèle LLM | section 2 | Un serveur Ollama local, les cartes des deux modèles, le décodage, le budget de sortie et les contrôles. |
| 3. Sortie générée | section 3 | Comment je lis les sorties, les modes de sortie, les itérations un bloc à la fois, le raisonnement et les folds. |
| 4. Évaluation | section 4 | Scores par fold face à LinearSVC, erreurs relues, coût et latence, test final, journal et conclusion. |


## 0. Mon problème et les acquis des TP1 et TP2

Je prédis le locuteur d'une réplique à partir de son seul texte, parmi 10 classes. Comme aux TP1 et TP2, je garde les répliques d'au moins 10 mots des 10 locuteurs les plus fréquents. Le modèle ne voit aucune donnée d'entraînement, sauf les exemples que je place dans le prompt.

| Fait observé dans les TP1 et TP2 | Ce que j'examine dans le TP3 |
|---|---|
| LinearSVC obtient un F1 macro de 0,482 ± 0,035 en validation croisée et 0,563 sur le test (TP1) | Un LLM sans entraînement atteint-il ce repère, et à quel coût ? |
| `kate`, `romeo`, `paulina` et `camillo` portent les poids les plus élevés (TP1, section 8.3) | Le LLM connaît-il déjà ces personnages et leur pièce, ou faut-il le lui dire ? |
| 9 confusions JULIET–ROMEO et 7 de GLOUCESTER vers DUKE VINCENTIO sur le test (TP1) | Les mêmes paires résistent-elles au LLM, et lesquelles corrige-t-il ? |
| Chaque modèle est ajusté dans les folds, jamais sur tout le développement (TP2) | Les exemples du prompt suivent la même règle, donc ils viennent du train du fold seulement. |
| Une prédiction est une classe parmi 10 (TP1 et TP2) | Une sortie générée est un texte libre, donc je dois la lire, la normaliser et compter les sorties hors liste. |
| Le TP2 comparait des architectures à données égales | Je compare ici deux modèles déjà entraînés, à budget mémoire comparable, sans toucher à leurs poids. |


> *Mes hypothèses :*
>
> - **H1, connaissance préalable.** Sans exemple, le LLM dépasse la classe majoritaire mais reste sous les 0,482 de LinearSVC, car le texte seul doit lui suffire pour reconnaître dix personnages. Testée dans les sections 3.1 et 3.6.
> - **H2, contexte.** Donner la pièce et le rôle de chaque personnage corrige surtout les répliques qui citent un nom. Le modèle relie un nom à une pièce plus sûrement qu'à un locuteur. Testée dans la section 3.4.
> - **H3, exemples.** Dix puis trente exemples tirés du train gagnent au moins 0,05 de F1 macro sur la meilleure version sans exemple, mais un autre tirage change le score. Testée dans les sections 3.4 et 3.6.
> - **H4, format.** Contraindre la sortie change la conformité plus que la qualité. Les sorties hors liste tombent donc à zéro sans écart de F1 macro au-delà de la dispersion des folds. Testée dans la section 3.3.
> - **H5, coût.** Le LLM coûte au moins cent fois la latence de LinearSVC par réplique. Il ne se justifie donc que si son F1 macro dépasse LinearSVC de plus que la dispersion des folds. Testée dans la section 4.3.
> - **H6, modèle à budget mémoire comparable.** Le grand modèle fortement quantifié dépasse le petit en F1 macro sur les mêmes prompts directs. Il coûte pourtant au moins deux fois sa latence par réplique. Testée dans les sections 3.6 et 4.3.
> - **H7, transfert du prompt.** Les gains obtenus en itérant sur un modèle se retrouvent sur l'autre avec le même signe, mais pas la même ampleur. Un prompt est en effet réglé contre le comportement d'un modèle. Testée dans les sections 3.4 et 3.6.
> - **H8, raisonnement.** Demander une sortie structurée peut déclencher un raisonnement interne, donc le mode de sortie et le raisonnement varient ensemble. Le raisonnement corrige surtout les répliques sans nom cité, au prix d'environ vingt fois plus de tokens générés. Testée dans les sections 3.3 et 3.5.

Je décide chaque version sur le jeu de mise au point, puis je confirme sur les cinq folds avant de regarder le test.


### Mon environnement

Je sélectionne `.venv/bin/python` comme kernel et j'installe les dépendances dans le terminal :

```sh
uv pip install --python .venv/bin/python -r requirements.txt
```

Les LLM tournent dans un serveur Ollama local, lancé à part dans le terminal :

```sh
ollama serve
ollama pull hf.co/unsloth/gemma-4-E4B-it-GGUF:UD-Q8_K_XL
ollama pull hf.co/unsloth/Qwen3.8-27B-GGUF:UD-Q2_K_XL
```

> *Mes choix :*
>
> - Un serveur local plutôt qu'une API hébergée, car les poids sont ouverts, aucune donnée ne sort de la machine et la version du modèle reste figée dans mon journal.
> - Deux modèles, un petit peu quantifié et un grand fortement quantifié, car ils occupent une place comparable en mémoire sur mon M3 Max. Leurs cartes sont lues dans la section 2.2.
> - Un seul des deux mène les itérations et l'autre vérifie les choix retenus, car mesurer chaque version sur les deux doublerait le temps sans changer les décisions. La section 2.5 fixe lequel mène.
> - Le client Python d'Ollama, figé dans `requirements.txt`, car chaque appel reste une requête JSON vers le serveur et le client me rend les compteurs de tokens et les durées.
> - Température 0 et seed 42, car une classification demande la génération la moins aléatoire, et je vérifie dans la section 2.4 que deux appels identiques donnent le même texte.
> - Un cache sur disque de chaque réponse, car chaque appel a un coût en temps et le notebook doit se rejouer de haut en bas sans relancer les mêmes appels.


In [1]:
# Environnement : versions, matériel, seed, client Ollama et présence des deux modèles.
import json
import platform
import random
import subprocess
import sys
import urllib.request
from importlib.metadata import version

import numpy as np
import ollama

# Seed fixée ici pour random et numpy, puis transmise au serveur à chaque appel (section 2.3).
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Serveur Ollama local, puis les deux modèles comparés
URL_OLLAMA = 'http://localhost:11434'
MODELES = {
    'gemma': 'hf.co/unsloth/gemma-4-E4B-it-GGUF:UD-Q8_K_XL',
    'qwen': 'hf.co/unsloth/Qwen3.8-27B-GGUF:UD-Q2_K_XL',
}
MODELE_DEV, MODELE_COMPARAISON = 'gemma', 'qwen'
client = ollama.Client(host=URL_OLLAMA)

# Versions et dépendances
print(f'Python : {platform.python_version()} — {sys.executable}')
for paquet in ('numpy', 'pandas', 'matplotlib', 'scikit-learn', 'ipykernel', 'ollama'):
    print(f'{paquet} : {version(paquet)}')

# Système et processeur
print(f'Système : {platform.platform()} — {platform.machine()}')
if sys.platform == 'darwin':
    processeur, ram = subprocess.check_output(
        ['sysctl', '-n', 'machdep.cpu.brand_string', 'hw.memsize'], text=True,
    ).splitlines()
    print(f'Processeur : {processeur} — RAM : {int(ram) / 2**30:g} Gio')

# Serveur : sa version, puis la présence des deux modèles et leur taille sur disque
with urllib.request.urlopen(f'{URL_OLLAMA}/api/version', timeout=10) as reponse:
    version_serveur = json.load(reponse)['version']
tailles = {modele.model: modele.size for modele in client.list().models}
for cle, nom in MODELES.items():
    assert nom in tailles, f'{nom} absent, à installer avec ollama pull {nom}'
    print(f'{cle} : {nom} — {tailles[nom] / 2**30:.2f} Gio')
print(f'Ollama : {version_serveur} — mise au point sur {MODELE_DEV} — Seed : {RANDOM_STATE}')

Python : 3.14.7 — /Users/hopy/Documents/ING3_GIA/ING3_IA_GB_S1/NLP/.venv/bin/python
numpy : 2.5.3
pandas : 3.0.5
matplotlib : 3.11.1
scikit-learn : 1.9.0
ipykernel : 7.3.0
ollama : 0.6.2
Système : macOS-26.5-arm64-arm-64bit-Mach-O — arm64
Processeur : Apple M3 Max — RAM : 36 Gio
gemma : hf.co/unsloth/gemma-4-E4B-it-GGUF:UD-Q8_K_XL — 9.04 Gio
qwen : hf.co/unsloth/Qwen3.8-27B-GGUF:UD-Q2_K_XL — 10.81 Gio
Ollama : 0.32.5 — mise au point sur gemma — Seed : 42


## 1. Prompting

### 1.1 Les mêmes répliques

Je reprends le chargement et le parser du TP1 comme au TP2, sans dépendre de leur état d'exécution. Le modèle reçoit la réplique brute, avec sa casse et sa ponctuation, car son tokenizer est dans la boîte noire et je ne le contrôle pas. Je ne prétraite rien, donc je mesure l'effet du prompt seul.


In [2]:
# Corpus : téléchargement de Tiny Shakespeare, parser du TP1 et jeu retenu, comme au TP2.
import re

import pandas as pd
from IPython.display import display

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SEUIL_MOTS, NB_LOCUTEURS = 10, 10

# Téléchargement du texte brut
with urllib.request.urlopen(URL, timeout=30) as reponse:
    texte = reponse.read().decode('utf-8')


def parser_repliques(txt: str) -> pd.DataFrame:
    """Reprend les règles du TP1 : un bloc, une étiquette et sa réplique."""
    lignes = []
    for bloc in txt.split("\n\n"):
        bloc = bloc.strip()
        if not bloc:
            continue
        parties = bloc.split("\n")
        entete = re.match(r"^([A-Z][A-Za-z' ]*):$", parties[0])  # Nom en majuscules puis « : »
        if entete and len(parties) > 1:
            lignes.append({
                'locuteur': entete.group(1).strip(),
                'replique': " ".join(p.strip() for p in parties[1:]).strip(),
            })
    return pd.DataFrame(lignes)


# Seuil de longueur avant de compter les locuteurs, ordre et indices du TP1 conservés
df = parser_repliques(texte)
longueurs = df['replique'].str.split().str.len()
locuteurs = df.loc[longueurs >= SEUIL_MOTS, 'locuteur'].value_counts().head(NB_LOCUTEURS).index
donnees = df.loc[(longueurs >= SEUIL_MOTS) & df['locuteur'].isin(locuteurs),
                 ['replique', 'locuteur']].copy()

Je contrôle ensuite le jeu retenu, car il doit être identique à celui des TP1 et TP2 pour que les scores restent comparables.

In [3]:
# Contrôles du jeu retenu : effectifs, longueurs, valeurs manquantes, doublons.
effectifs = donnees['locuteur'].value_counts()
longueurs_retenues = donnees['replique'].str.split().str.len()

# Comptes globaux
print(f'Extraction : {len(df):,} répliques, {df["locuteur"].nunique()} labels')
print(f'Jeu retenu : {len(donnees)} répliques, {len(effectifs)} classes')
print(f'Longueurs (mots) : min {longueurs_retenues.min()}, '
      f'médiane {longueurs_retenues.median():g}, max {longueurs_retenues.max()}')
print(f'Valeurs manquantes : {donnees.isna().sum().sum()}')
print(f'Répliques vides : {donnees["replique"].str.strip().eq("").sum()}')
print(f'Textes dupliqués dans le jeu retenu : {donnees["replique"].duplicated().sum()}')
display(effectifs.rename('Répliques').to_frame())

Extraction : 7,094 répliques, 297 labels
Jeu retenu : 962 répliques, 10 classes
Longueurs (mots) : min 10, médiane 31, max 578
Valeurs manquantes : 0
Répliques vides : 0
Textes dupliqués dans le jeu retenu : 0


,Répliques
locuteur,
GLOUCESTER,129
DUKE VINCENTIO,125
MENENIUS,107
PETRUCHIO,95
ROMEO,94
CORIOLANUS,87
JULIET,85
LEONTES,82
ISABELLA,80


> *Mes observations :*
>
> Je retrouve les 962 répliques des TP1 et TP2, avec 7 094 répliques et 297 étiquettes à l'extraction, puis 10 classes de 78 à 129 répliques. Aucune valeur manquante, aucune réplique vide et aucun doublon. Les longueurs vont de 10 à 578 mots, avec une médiane de 31, et le modèle recevra chaque réplique telle quelle.

### 1.2 Partitions, folds et jeu de mise au point

Je garde le découpage 80/20 stratifié avec la seed 42 et les cinq folds du TP2, car chaque version de prompt sera comparée réplique par réplique à LinearSVC. Le test ne sert qu'une fois, dans la section 4.4.

In [4]:
# Découpage développement / test du TP1, 80/20 stratifié, seed 42, puis les cinq folds du TP2.
from sklearn.model_selection import StratifiedKFold, train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    donnees['replique'], donnees['locuteur'],
    test_size=0.2, stratify=donnees['locuteur'], random_state=RANDOM_STATE,
)
assert set(X_train.index).isdisjoint(X_test.index)  # Aucun indice partagé
assert not set(X_train).intersection(X_test)  # Aucun texte partagé

# Cinq folds stratifiés, indices d'origine utilisables avec .loc
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = [{'train': X_train.index[train], 'validation': X_train.index[validation]}
         for train, validation in cv.split(X_train, y_train)]
assert all(y_train.loc[fold['validation']].nunique() == y_train.nunique() for fold in folds)

# Tailles, empreinte du découpage, puis taille des folds
print(f'Développement : {len(X_train)} répliques, test : {len(X_test)}')
print(f'Premiers indices du test : {list(X_test.index[:10])}')
display(pd.DataFrame([{nom: len(indices) for nom, indices in fold.items()} for fold in folds],
                     index=pd.RangeIndex(1, len(folds) + 1, name='Fold')))

Développement : 769 répliques, test : 193
Premiers indices du test : [3009, 6783, 5903, 1321, 3338, 2294, 3344, 2875, 3378, 2212]


,train,validation
Fold,,
1,615,154
2,615,154
3,615,154
4,615,154
5,616,153


J'ajoute un jeu de mise au point de 5 répliques par classe, tirées parmi les 769 répliques de développement avec la seed 42. Chaque appel au modèle coûte du temps, mesuré dans la section 2.5, et je prévois plusieurs versions, donc j'itère sur ces 50 répliques. Les 719 répliques restantes forment une réserve pour les exemples du prompt, donc aucune réplique évaluée n'apparaît dans un prompt. Une réplique pèse 2 points sur 50. Je fixe donc un gain net minimal de 3 répliques dans la section 1.4, puis je confirme sur les cinq folds dans la section 3.6.

In [5]:
# Jeu de mise au point : 5 répliques par classe parmi les 769, seed 42, et réserve des 719 autres.
PAR_CLASSE = 5
MISE_AU_POINT = (y_train.groupby(y_train).sample(PAR_CLASSE, random_state=RANDOM_STATE)
                 .index.sort_values())
RESERVE = X_train.index.difference(MISE_AU_POINT)

# Vérifications : 5 par classe, disjonction et réserve complémentaire
assert (y_train.loc[MISE_AU_POINT].value_counts() == PAR_CLASSE).all()
assert not MISE_AU_POINT.isin(RESERVE).any()
assert len(MISE_AU_POINT) + len(RESERVE) == len(X_train)

# Tailles, répartition dans les folds et longueur face aux 769
fold_de = pd.Series({indice: numero for numero, fold in enumerate(folds, start=1)
                     for indice in fold['validation']})
mots = X_train.str.split().str.len()
print(f'Mise au point : {len(MISE_AU_POINT)} répliques, réserve : {len(RESERVE)}')
print(f'Répliques de mise au point par fold de validation : '
      f'{fold_de.loc[MISE_AU_POINT].value_counts().sort_index().to_dict()}')
print(f'Longueur médiane (mots) : {mots.loc[MISE_AU_POINT].median():g} sur les 50, '
      f'{mots.median():g} sur les 769')

Mise au point : 50 répliques, réserve : 719
Répliques de mise au point par fold de validation : {1: 9, 2: 14, 3: 4, 4: 15, 5: 8}
Longueur médiane (mots) : 27 sur les 50, 31 sur les 769


> *Mes observations :*
>
> Le découpage redonne 769 répliques de développement et 193 de test, avec les mêmes premiers indices de test qu'au TP2, et les folds gardent leurs 615 ou 616 répliques de train. Le jeu de mise au point compte bien 50 répliques, 5 par classe, et la réserve les 719 autres. Les 50 se répartissent inégalement entre les folds de validation, de 4 à 15 répliques. Cela ne gêne pas, car je ne lis que leurs prédictions hors fold. Leur longueur médiane de 27 mots, contre 31, les rend un peu plus courtes que l'ensemble.

Je reprends les repères du TP1 sur les cinq folds avec l'évaluateur du TP2, la classe majoritaire et le pipeline TF–IDF + LinearSVC exact du TP1. J'attends 0,482 ± 0,035 de F1 macro, ce qui vérifie que les folds sont les mêmes. Je garde les prédictions par réplique, car la section 4.1 compare chaque version de prompt à LinearSVC sur les mêmes répliques. Je lis aussi son score sur les 50 répliques de mise au point, pour situer chaque version pendant les itérations.


In [6]:
# Outils d'évaluation du TP2 : scores, registre et évaluateur ML sur les cinq folds.
import time

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

PARAMS_TFIDF = dict(ngram_range=(1, 2), min_df=2, max_df=1.0, stop_words=None, sublinear_tf=True)
PARAMS_SVC = dict(C=1.0, class_weight=None, max_iter=5000, random_state=42)  # Réglages du TP1
resultats = {}  # nom -> scores par fold et prédictions de validation par indice


def pourcentage(partie: float, total: float) -> float:
    """Part en pourcentage."""
    return 100 * partie / total


def scores(vrais, predits) -> dict:
    """F1 macro et accuracy, les deux métriques de comparaison."""
    return {'F1 macro': f1_score(vrais, predits, average='macro', zero_division=0),
            'Accuracy': accuracy_score(vrais, predits)}


def resumer(nom: str) -> pd.DataFrame:
    """Moyenne et écart-type entre folds des scores d'un modèle du registre."""
    table = resultats[nom]['folds'][['F1 train', 'F1 macro', 'Accuracy', 'Temps (s)']]
    return pd.DataFrame({'moyenne': table.mean(), 'écart-type': table.std()}).T.round(3)


def evaluer_ml(nom: str, construire_pipeline) -> pd.DataFrame:
    """Ajuste un pipeline neuf sur chaque fold, garde scores, temps et prédictions par indice."""
    lignes, predictions = [], pd.Series(index=X_train.index, dtype=object)
    for numero, fold in enumerate(folds, start=1):
        debut = time.perf_counter()
        textes, cibles = X_train.loc[fold['train']], y_train.loc[fold['train']]
        pipeline = construire_pipeline().fit(textes, cibles)
        predit = pipeline.predict(X_train.loc[fold['validation']])
        predictions.loc[fold['validation']] = predit
        f1_train = scores(cibles, pipeline.predict(textes))['F1 macro']
        lignes.append({'Fold': numero, 'F1 train': f1_train,
                       **scores(y_train.loc[fold['validation']], predit),
                       'Temps (s)': time.perf_counter() - debut})
    resultats[nom] = {'famille': 'ML', 'folds': pd.DataFrame(lignes).set_index('Fold'),
                      'predictions': predictions}
    return resultats[nom]['folds'].round(3)


def construire_svc_tp1() -> Pipeline:
    """Pipeline exact du TP1 : TF–IDF par défaut de scikit-learn puis LinearSVC."""
    return Pipeline([('tfidf', TfidfVectorizer(lowercase=True, **PARAMS_TFIDF)),
                     ('svc', LinearSVC(**PARAMS_SVC))])

Je lance les deux repères sur les cinq folds, puis je lis le score de LinearSVC sur les 50 répliques de mise au point.

In [7]:
# Repères ML sur les cinq folds : classe majoritaire et LinearSVC du TP1, puis LinearSVC sur les 50.
REPERES = {'Majoritaire': lambda: DummyClassifier(strategy='most_frequent'),
           'LinearSVC TP1': construire_svc_tp1}
for nom, construire in REPERES.items():
    print(nom)
    display(evaluer_ml(nom, construire))
display(pd.concat({nom: resumer(nom) for nom in REPERES}))

# Sur les 50, LinearSVC garde ses prédictions hors fold, donc aucune réplique vue à l'entraînement
predit_50 = resultats['LinearSVC TP1']['predictions'].loc[MISE_AU_POINT]
score_50 = scores(y_train.loc[MISE_AU_POINT], predit_50)
print(f'LinearSVC TP1 sur les 50 répliques de mise au point : F1 macro {score_50["F1 macro"]:.3f}, '
      f'accuracy {score_50["Accuracy"]:.3f}, '
      f'{(predit_50 == y_train.loc[MISE_AU_POINT]).sum()} justes sur 50')

Majoritaire


,F1 train,F1 macro,Accuracy,Temps (s)
Fold,,,,
1,0.024,0.024,0.136,0.006
2,0.024,0.024,0.136,0.005
3,0.024,0.024,0.136,0.005
4,0.024,0.023,0.130,0.005
5,0.024,0.023,0.131,0.005


LinearSVC TP1


,F1 train,F1 macro,Accuracy,Temps (s)
Fold,,,,
1,1.0,0.453,0.461,0.070
2,1.0,0.429,0.435,0.069
3,1.0,0.500,0.513,0.068
4,1.0,0.511,0.519,0.067
5,1.0,0.517,0.529,0.068


F1 train  F1 macro  Accuracy  Temps (s)
Majoritaire   moyenne        0.024     0.024     0.134      0.005
              écart-type     0.000     0.001     0.003      0.001
LinearSVC TP1 moyenne        1.000     0.482     0.492      0.069
              écart-type     0.000     0.039     0.041      0.001

LinearSVC TP1 sur les 50 répliques de mise au point : F1 macro 0.303, accuracy 0.320, 16 justes sur 50


> *Mes observations :*
>
> LinearSVC du TP1 retrouve 0,482 de F1 macro et 0,492 d'accuracy, avec les mêmes scores par fold qu'au TP2, donc les folds sont les mêmes. Son écart-type vaut 0,039 et non 0,035, car `resumer` divise par n − 1 comme au TP2. Le repère majoritaire reste à 0,024. Sur les 50 répliques de mise au point, LinearSVC ne trouve que 16 locuteurs, soit 0,303 de F1 macro et 0,320 d'accuracy.
>
> *Ma conclusion :* les 50 répliques sont plus difficiles que la moyenne pour LinearSVC, peut-être parce qu'elles sont plus courtes, et une réplique y pèse 2 points. Je compare donc chaque version de prompt à ce 0,303 pendant les itérations, puis aux 0,482 des folds dans la section 3.6.

### 1.3 Le prompt en blocs

Un prompt est une spécification testable, donc je l'écris en six blocs nommés, `objectif`, `contexte`, `contraintes`, `format`, `exemples` et `criteres`. Chaque bloc est une chaîne, vide au départ sauf l'objectif, et une fonction les assemble en messages. Le rôle system porte les règles stables, le rôle user porte les données de la tâche, donc je peux changer la demande sans réécrire les règles.

La réplique est délimitée par des guillemets, car elle est une donnée et non une instruction. Elle vient en dernier, car le préfixe commun à tous les appels peut alors rester en cache, ce que je vérifie dans la section 2.4. Une version est un dictionnaire avec son parent, le bloc changé, sa valeur, son hypothèse et sa décision, dans un registre `VERSIONS`. Chaque changement reste ainsi traçable, comme une clé de `REGLAGES` au TP2. J'écris les blocs en anglais, car les répliques sont en anglais et le modèle y est le plus sûr.


In [8]:
# Le prompt en six blocs, le registre des versions, puis v0, la version zero-shot minimale.
# Ordre d'assemblage des blocs dans le prompt
BLOCS = ['objectif', 'contexte', 'criteres', 'contraintes', 'format', 'exemples']
NOMS = sorted(locuteurs)  # Liste fermée des dix locuteurs
VERSIONS = {}  # nom -> parent, bloc changé, blocs complets, hypothèse et décision


def declarer_version(nom: str, parent, bloc: str, valeur, hypothese: str) -> dict:
    """Copie les blocs du parent et en change un seul, ou fixe les blocs de départ de v0."""
    blocs = dict(VERSIONS[parent]['blocs']) if parent else dict.fromkeys(BLOCS, '')
    changements = valeur if isinstance(valeur, dict) else {bloc: valeur}
    assert set(changements) <= set(BLOCS), changements
    blocs.update(changements)
    VERSIONS[nom] = {'parent': parent, 'bloc': bloc, 'blocs': blocs, 'hypothese': hypothese,
                     'decision': None}
    return VERSIONS[nom]


def formater_exemples(paires: list) -> str:
    """Exemples réplique puis locuteur, dans le même format que la réplique à classer."""
    return '\n\n'.join(f'Reply: "{replique}"\nSpeaker: {locuteur}' for replique, locuteur in paires)


def assembler_messages(version: dict, replique: str, exemples: list | None = None) -> list:
    """System avec les blocs stables, user avec les exemples puis la réplique, en dernier."""
    blocs = version['blocs']
    systeme = '\n\n'.join(blocs[bloc] for bloc in BLOCS if bloc != 'exemples' and blocs[bloc])
    utilisateur = []
    if exemples:
        utilisateur += [blocs['exemples'], formater_exemples(exemples)]
    utilisateur.append(f'Reply: "{replique}"\nSpeaker:')
    return [{'role': 'system', 'content': systeme},
            {'role': 'user', 'content': '\n\n'.join(part for part in utilisateur if part)}]


declarer_version('v0', None, 'objectif, contraintes', {
    'objectif': ('Identify which Shakespeare character speaks the reply. The speaker is one of: '
                 + ', '.join(NOMS) + '.'),
    'contraintes': 'Answer with one name from the list.',
}, 'H1')
print(f'{len(NOMS)} noms dans la liste fermée, versions déclarées : {list(VERSIONS)}')

10 noms dans la liste fermée, versions déclarées : ['v0']


Je regarde enfin les messages de v0 tels que le modèle les recevra, pour une réplique de mise au point.

In [9]:
# Messages de v0 pour la première réplique de mise au point, puis un exemple de la réserve.
premiere = MISE_AU_POINT[0]
for message in assembler_messages(VERSIONS['v0'], X_train.loc[premiere]):
    print(f"[{message['role']}]\n{message['content']}\n")
print(f'Locuteur réel : {y_train.loc[premiere]}\n')
print('Un exemple de la réserve, au format que prendront les exemples du prompt :')
print(formater_exemples([(X_train.loc[RESERVE[0]], y_train.loc[RESERVE[0]])]))

[system]
Identify which Shakespeare character speaks the reply. The speaker is one of: CORIOLANUS, DUKE VINCENTIO, GLOUCESTER, ISABELLA, JULIET, KING RICHARD II, LEONTES, MENENIUS, PETRUCHIO, ROMEO.

Answer with one name from the list.

[user]
Reply: "The senators of Rome are this good belly, And you the mutinous members; for examine Their counsels and their cares, digest things rightly Touching the weal o' the common, you shall find No public benefit which you receive But it proceeds or comes from them to you And no way from yourselves. What do you think, You, the great toe of this assembly?"
Speaker:

Locuteur réel : MENENIUS

Un exemple de la réserve, au format que prendront les exemples du prompt :
Reply: "What work's, my countrymen, in hand? where go you With bats and clubs? The matter? speak, I pray you."
Speaker: MENENIUS


> *Mes observations :*
>
> Le message system de v0 ne contient que l'objectif, avec la liste fermée des dix noms, et la contrainte d'un seul nom. Le message user porte la réplique entre guillemets, puis « Speaker: », le même format que les exemples de la réserve. Aucun contexte, critère ou format n'est donné, donc v0 mesure ce que le modèle sait déjà de ces personnages. Le message system reste identique pour toutes les répliques, et seule la fin du message user change, ce qui garde un préfixe commun en cache.

### 1.4 L'échelle d'itérations et la règle de décision

Je planifie une échelle de versions, un seul bloc changé par version, car un changement isolé est le seul que je peux attribuer. L'échelle est un plan et non une consigne, donc les erreurs lues à chaque étape choisissent le bloc suivant.

| Version | Bloc changé | Hypothèse | Ce que je regarde |
|---|---|---|---|
| v0 | aucun, zero-shot minimal | H1 | sorties brutes, conformité, F1 macro sur les 50 |
| v1 | `format`, trois modes (texte strict, JSON demandé, schéma imposé) | H4, H8 | conformité, troncature, tokens de raisonnement, latence |
| v2 | `criteres` (indices à utiliser, un nom cité n'est pas le locuteur) | les erreurs à nom cité | répliques corrigées et cassées |
| v3 | `contexte` (pièce et rôle des dix personnages) | H2 | erreurs par pièce |
| v4 | `exemples`, un par classe, tirés de la réserve | H3 | gain net, tokens du prompt |
| v5 | `exemples`, trois par classe | H3, H5 | gain net, coût, cache de préfixe |
| v6 | choix des exemples, répliques que LinearSVC rate | les cas frontière guident mieux | gain net face à v4 et v5 |
| v7, dans la section 3.5 | `format`, raisonnement demandé puis locuteur | H8 | gain net, tokens de raisonnement |

Chaque itération suit la même boucle. Je déclare la version depuis la dernière retenue, je l'évalue sur les 50 répliques, puis je la compare à son parent en répliques corrigées, cassées et gain net. Je lis les confusions les plus fréquentes et cinq erreurs avec leur sortie brute, je décide, puis j'écris la décision dans `VERSIONS` et une ligne dans le journal.

> *Mes choix :*
>
> - Une version remplace son parent si son gain net atteint 3 répliques sur 50, si sa conformité ne baisse pas et si son prompt tient dans `BUDGET_TOKENS`.
> - Sinon le parent reste et la version garde sa ligne dans le journal, car une hypothèse réfutée reste une information.
> - Les exemples sont équilibrés par classe, car des exemples déséquilibrés copieraient le déséquilibre des classes dans le prompt.
> - J'itère sur un seul modèle et je vérifie le transfert sur l'autre aux paliers retenus, car régler un prompt contre deux modèles à la fois empêche d'attribuer un gain.


In [10]:
# Règle de décision de l'échelle : bilan d'une version face à son parent, puis décision.
SEUIL_GAIN = 3  # Répliques nettes sur 50, soit 6 points d'accuracy


def bilan_changement(vrais, predit_parent, predit_version) -> dict:
    """Répliques corrigées et cassées par une version face à son parent, et gain net."""
    vrais, parent, version = map(np.asarray, (vrais, predit_parent, predit_version))
    corrigees = int(((version == vrais) & (parent != vrais)).sum())
    cassees = int(((version != vrais) & (parent == vrais)).sum())
    return {'Corrigées': corrigees, 'Cassées': cassees, 'Gain net': corrigees - cassees}


def regle_decision(gain_net: int, conformite_version: float, conformite_parent: float,
                   tokens_prompt: int, budget: int) -> tuple:
    """La version remplace son parent si les trois conditions de la section 1.4 tiennent."""
    if gain_net < SEUIL_GAIN:
        return 'parent gardé', f'gain net {gain_net} < {SEUIL_GAIN}'
    if conformite_version < conformite_parent:
        return 'parent gardé', 'conformité en baisse'
    if tokens_prompt > budget:
        return 'parent gardé', f'prompt de {tokens_prompt} tokens > budget de {budget}'
    return 'version retenue', f'gain net {gain_net}, conformité et budget tenus'


# Exemple construit sur 8 répliques : la version corrige 4 erreurs du parent et en casse 1
vrais = ['ROMEO', 'JULIET', 'ROMEO', 'GLOUCESTER', 'ISABELLA', 'LEONTES', 'MENENIUS', 'PETRUCHIO']
parent = ['ROMEO', 'ROMEO', 'JULIET', 'DUKE VINCENTIO', 'CORIOLANUS', 'LEONTES', 'MENENIUS',
          'PETRUCHIO']
version = ['ROMEO', 'JULIET', 'ROMEO', 'GLOUCESTER', 'ISABELLA', 'LEONTES', 'CORIOLANUS',
           'PETRUCHIO']
bilan = bilan_changement(vrais, parent, version)
print(bilan)
for conformite, tokens in ((1.0, 300), (0.96, 300), (1.0, 5000)):
    print(f'conformité {conformite}, prompt de {tokens} tokens, budget fictif de 2000 : '
          f'{regle_decision(bilan["Gain net"], conformite, 0.98, tokens, 2000)}')

{'Corrigées': 4, 'Cassées': 1, 'Gain net': 3}
conformité 1.0, prompt de 300 tokens, budget fictif de 2000 : ('version retenue', 'gain net 3, conformité et budget tenus')
conformité 0.96, prompt de 300 tokens, budget fictif de 2000 : ('parent gardé', 'conformité en baisse')
conformité 1.0, prompt de 5000 tokens, budget fictif de 2000 : ('parent gardé', 'prompt de 5000 tokens > budget de 2000')


> *Mes observations :*
>
> Sur l'exemple construit, la version corrige 4 erreurs du parent et en casse 1, soit un gain net de 3. Elle est retenue avec une conformité de 1, puis refusée quand sa conformité baisse à 0,96 ou quand son prompt dépasse le budget. La règle est donc fixée avant le premier résultat du modèle, et `BUDGET_TOKENS` viendra de la mesure de la section 2.3.

## 2. Modèle LLM

### 2.1 Un serveur local et une boîte noire

Je contrôle les entrées et les paramètres, puis le serveur exécute le modèle et renvoie une sortie probabiliste. Le client envoie une requête JSON avec les messages par rôle, les options de décodage et le format attendu. La réponse peut porter deux canaux distincts, le contenu et un raisonnement, donc je garde les deux.

Chaque réponse est mise en cache sur disque dans `cache_llm/`, sous une clé qui hache le modèle, les messages, les options et le format. Le modèle fait partie de la clé, donc les deux modèles ne se mélangent jamais. Un nouveau passage du notebook rejoue ainsi les mêmes sorties, et le cache garde les compteurs de tokens et les durées, donc le coût de chaque version reste lisible.


In [7]:
# TODO : fixer DOSSIER_CACHE et écrire cle_cache(corps), un hachage SHA-256 du modèle, des messages,
#   des options et du format.
# TODO : écrire appeler_llm(modele, messages, options, format=None, cache=True) qui lit le cache ou
#   appelle client.chat, puis enregistre la réponse en JSON.
# TODO : écrire extraire_mesures(reponse), contenu, raisonnement, tokens du prompt, tokens générés,
#   durées en secondes, raison d'arrêt et provenance du cache.
# TODO : afficher un appel sur une réplique de mise au point.

### 2.2 Les cartes des deux modèles

Je lis la carte de chaque modèle depuis le serveur, architecture, nombre de paramètres, quantification, taille de contexte, taille sur disque et capacités déclarées. Le choix d'un modèle inclut son mode d'accès et ses limites, donc ces lignes cadrent toute la comparaison.

Je lis la quantification annoncée par le serveur plutôt que celle du nom de l'image, car les deux peuvent différer. Je regarde aussi si un mode de réflexion natif est déclaré, car tous les modèles n'en offrent pas, et un raisonnement doit alors être demandé dans le prompt.


In [8]:
# TODO : afficher client.show pour les deux modèles en un tableau, famille, paramètres,
#   quantification, contexte et capacités.
# TODO : ajouter la taille sur disque de chaque modèle et le nom de l'image, pour comparer les deux
#   quantifications annoncées.

> *Mes observations :*
>
> *Ma conclusion :*


### 2.3 Décodage et budget de sortie

Je fixe les options de décodage dans un dictionnaire `OPTIONS` par modèle, comme `REGLAGES` au TP2, donc une expérience n'en change qu'une clé. Température 0, car une classification demande la génération la moins aléatoire. Seed 42, pour rejouer un appel à l'identique. Je laisse top-p et top-k par défaut, car je ne règle qu'un paramètre à la fois.

`num_predict` borne la longueur de la sortie, et une borne trop courte coupe la réponse avant sa fin. Je la fixe donc depuis la distribution mesurée des tokens générés, par mode et par modèle, avec une marge, et non depuis la longueur d'une étiquette. Je surveille la raison d'arrêt de chaque appel, car une génération coupée rend un contenu vide que mon parseur lirait à tort comme une sortie hors liste.

La fenêtre de contexte est un budget partagé entre les instructions, les exemples, la réplique, la sortie et une marge. Je mesure les tokens du prompt de v0 sur les 50 répliques, puis j'estime le coût de dix et de trente exemples. J'en déduis `num_ctx` et `BUDGET_TOKENS` pour la règle de la section 1.4.


In [9]:
# TODO : définir OPTIONS par modèle, température 0, seed 42, num_predict et num_ctx.
# TODO : mesurer la distribution des tokens générés par mode et par modèle, puis fixer num_predict
#   avec une marge, et vérifier qu'aucun appel ne s'arrête sur la borne de longueur.
# TODO : mesurer les tokens du prompt de v0 sur MISE_AU_POINT, minimum, médiane et maximum.
# TODO : estimer les tokens de dix et de trente exemples depuis la longueur des répliques de
#   RESERVE.
# TODO : fixer BUDGET_TOKENS depuis ces mesures.

> *Mes observations :*
>
> *Ma conclusion :*


### 2.4 Contrôles avant d'itérer

Trois contrôles précèdent la première itération, car une mesure de prompt n'a de sens que si l'appel est reproductible et si son coût est connu. Je les passe sur les deux modèles, car rien ne garantit qu'ils se comportent pareil.

Le déterminisme, deux appels identiques hors cache, à température 0 puis à 0,7 avec la seed, doivent rendre le même texte. Le cache de préfixe, un prompt à trente exemples envoyé deux fois puis avec une autre réplique, doit réduire le temps d'évaluation du prompt. La latence par appel, chargement exclu, donne le budget d'une version. Les deux premiers contrôles sont les lignes 1 et 2 du journal.


In [10]:
# TODO : appeler deux fois le même prompt hors cache, à température 0 puis 0,7 avec la seed, et
#   comparer, sur les deux modèles.
# TODO : envoyer un prompt à trente exemples deux fois puis avec une autre réplique, et lire les
#   durées d'évaluation du prompt, sur les deux modèles.
# TODO : mesurer la latence par appel hors chargement, sur les deux modèles, et afficher un tableau.

> *Mes observations :*
>
> *Ma conclusion :*


### 2.5 Le modèle de mise au point et la règle de bascule

Charger un modèle coûte du temps, donc je ne mélange jamais les deux dans une même boucle. Je groupe tous les appels d'un modèle, puis je passe à l'autre, car alterner à chaque réplique paierait un chargement par appel.

Je désigne un modèle de mise au point, qui mène les itérations de la section 3.4, et un modèle de comparaison, qui vérifie les paliers retenus. Le tableau ci-dessous mesure le coût de bascule et la latence des deux, puis justifie lequel mène. Je traduis ces latences en budget pour 50, 769 et 193 appels, car ce budget décide de ce que je peux passer sur les folds.


In [11]:
# TODO : mesurer le coût de bascule, un appel sur chaque modèle en alternance, chargement compris.
# TODO : mesurer la latence par appel des deux modèles sur les mêmes répliques, hors chargement, en
#   mode direct puis avec trente exemples.
# TODO : afficher le budget de 50, 769 et 193 appels par modèle et par mode, puis justifier
#   MODELE_DEV.
# TODO : écrire grouper_par_modele(travaux) qui exécute tous les appels d'un modèle avant de
#   basculer.

> *Mes observations :*
>
> *Ma conclusion :*


## 3. Sortie générée

### 3.1 Première génération avec v0

> *Mon hypothèse :* sans exemple, v0 dépasse la classe majoritaire mais reste sous LinearSVC sur les 50 répliques, car le texte seul doit suffire à reconnaître dix personnages. C'est un premier test de H1.

Je lance v0 sur les 50 répliques de mise au point et j'affiche cinq sorties brutes avec le locuteur réel. Je regarde le texte, sa longueur et la raison d'arrêt, car une sortie générée n'est pas encore une prédiction.


In [12]:
# TODO : écrire evaluer_version(nom, modele, indices, exemples=None) qui renvoie un DataFrame par
#   réplique, sortie brute, raisonnement, tokens, raison d'arrêt et durée, rangé dans ESSAIS.
# TODO : lancer v0 sur MISE_AU_POINT et afficher cinq sorties brutes avec le locuteur réel.

> *Mes observations :*
>
> *Ma conclusion :*


### 3.2 Lire la sortie générée

Une sortie est un texte libre, donc je la normalise avant de la compter. La règle nettoie les espaces et la casse, puis accepte un nom exact ou un seul nom de la liste contenu dans le texte.

Je distingue trois issues, car elles ne se corrigent pas de la même façon :
- une étiquette valide, lue comme un nom de la liste,
- une sortie hors liste, lue mais sans nom reconnaissable,
- une sortie tronquée, arrêtée par la borne de longueur avant sa fin.

Une troncature est un défaut de budget et non une erreur du modèle, donc je la compte à part et je corrige `num_predict`. Les deux autres issues comptent comme des erreurs dans le F1 macro, et la sortie hors liste reçoit sa ligne dans la matrice de confusion. Je compare ensuite v0 à LinearSVC sur les mêmes 50 répliques, en score et par paire de locuteurs confondus.


In [13]:
# TODO : écrire normaliser_reponse(texte), nom exact, puis nom unique contenu, sinon hors liste.
# TODO : marquer les sorties tronquées depuis la raison d'arrêt et les compter à part des hors
#   liste.
# TODO : écrire scorer(nom), F1 macro, accuracy, conformité et taux de troncature, puis
#   erreurs_par_paire(nom).
# TODO : scorer v0 et LinearSVC sur MISE_AU_POINT, puis afficher les paires les plus confondues.

> *Mes observations :*
>
> *Ma conclusion :*


### 3.3 Trois modes de sortie

> *Mon hypothèse :* contraindre la sortie change la conformité plus que la qualité, donc les sorties hors liste disparaissent sans gain net de F1 macro. C'est le test de H4.

Je déclare trois versions v1 depuis v0, un seul bloc `format` changé. Un texte strict, un nom exact et rien d'autre. Un JSON demandé dans le prompt, puis lu par mon parseur. Un schéma JSON imposé au serveur, avec la liste des dix noms comme énumération.

Un modèle peut décider de raisonner quand la sortie est contrainte, donc le mode et le raisonnement changeraient ensemble. Je mesure alors les deux, conformité et troncature d'un côté, tokens de raisonnement de l'autre, et je le dis avant de conclure sur H4. Je teste les trois modes sur les deux modèles, car la fiabilité d'un mode contraint est une propriété du modèle et non du prompt. Chaque modèle garde ensuite le mode qui lui réussit.


In [14]:
# TODO : déclarer v1-texte, v1-json et v1-schema depuis v0, bloc format seul.
# TODO : évaluer les trois modes sur les deux modèles, sur MISE_AU_POINT, en groupant par modèle.
# TODO : afficher conformité, troncature, F1 macro, tokens de réponse, tokens de raisonnement et
#   latence.
# TODO : appliquer decider par modèle, enregistrer la décision dans VERSIONS et fixer RETENUE par
#   modèle.

> *Mes observations :*
>
> *Ma conclusion :*


### 3.4 Itérer, un bloc à la fois

Chaque itération suit la boucle de la section 1.4. Je déclare la version depuis `RETENUE` avec un seul bloc changé et son hypothèse, je l'évalue sur les 50 répliques, puis je la compare à son parent. Je lis les confusions les plus fréquentes et cinq répliques en erreur avec leur sortie brute, car les erreurs choisissent le bloc suivant. J'applique la règle, j'écris la décision dans `VERSIONS` et une ligne dans le journal.

Les exemples de v4 à v6 viennent de la réserve de 719 répliques, jamais des 50 évaluées. Avant v3, je demande au modèle la pièce de chaque personnage en dix appels, puis je compare à la liste que je fournis. H2 suppose en effet qu'il relie un nom à une pièce. Je n'optimise pas sur quelques anecdotes, donc un gain net de 3 répliques est mon minimum et les cinq folds tranchent ensuite.

À chaque palier retenu, je rejoue la version sur le modèle de comparaison, sur les mêmes 50 répliques. Le gain garde-t-il son signe et son ampleur ? Cette colonne de transfert teste H7 pour 50 appels seulement, car un prompt réglé contre un modèle peut n'apporter rien à un autre.


In [15]:
# TODO : une cellule par version à l'implémentation, v2 puis v3, v4, v5 et v6, chacune depuis
#   RETENUE.
# TODO : avant v3, demander au modèle la pièce de chaque personnage en dix appels et comparer à ma
#   liste.
# TODO : pour chaque version, declarer_version, evaluer_version, scorer, comparer_au_parent, puis
#   afficher erreurs_par_paire et cinq répliques en erreur avec leur sortie brute.
# TODO : decider, écrire la décision dans VERSIONS et mettre à jour RETENUE.
# TODO : à chaque palier retenu, rejouer la version sur le modèle de comparaison et noter le
#   transfert.
# TODO : afficher le registre complet, une ligne par version avec score, conformité, coût, transfert
#   et décision.

> *Mes observations :*
>
> *Ma conclusion :*


### 3.5 Faire raisonner le modèle

> *Mon hypothèse :* le raisonnement corrige surtout les répliques qui ne citent aucun nom, car c'est là que l'indice lexical manque. C'est le test de H8, et c'est la version v7 de mon échelle.

Trois façons d'obtenir une étiquette, directement, en demandant un raisonnement dans le bloc `format`, puis par le drapeau de réflexion du serveur quand le modèle l'annonce. Un modèle qui refuse ce drapeau le dit par une erreur, que j'affiche plutôt que de la contourner.

Je compare ces façons sur les 50 répliques et je sépare les tokens de raisonnement des tokens de réponse, car les deux ne coûtent pas la même chose. Je lis enfin le gain séparément sur les répliques qui citent un nom de la liste et sur les autres.


In [16]:
# TODO : déclarer les trois configurations, directe, raisonnement demandé par le bloc format, et
#   réflexion native par le drapeau du serveur.
# TODO : tenter le drapeau sur les deux modèles et afficher le message d'erreur du modèle qui le
#   refuse.
# TODO : évaluer chaque configuration disponible sur MISE_AU_POINT, pour les deux modèles, groupé
#   par modèle.
# TODO : afficher F1 macro, tokens de réponse, tokens de raisonnement, troncatures et latence.
# TODO : séparer le gain sur les répliques qui citent un nom de la liste et sur les autres.

> *Mes observations :*
>
> *Ma conclusion :*


### 3.6 Les versions retenues sur les cinq folds

Je passe v0 et la version retenue sur les cinq folds, pour les deux modèles, car 50 répliques ne suffisent pas pour conclure. Je ne garde que les configurations sans raisonnement, car le budget mesuré dans la section 2.5 rend une configuration qui raisonne trop longue sur 769 répliques. Les conclusions de la section 3.5 restent donc appuyées sur 50 répliques, et je le signale sans le corriger.

Les exemples d'un fold viennent de son train seulement, tirés une fois par fold avec la seed 42, donc chaque réplique de validation est prédite sans avoir été vue dans un prompt. Je groupe les appels par modèle, puis je rassemble les 769 prédictions comme au TP2. Le jeu de mise au point a guidé mes choix, donc je donne aussi le score sur les 719 répliques qu'aucune itération n'a regardées.


In [17]:
# TODO : écrire exemples_du_fold(fold, version, seed) depuis fold['train'] seulement, avec l'assert
#   de disjonction avec fold['validation'].
# TODO : écrire evaluer_llm_folds(nom, modele) qui écrit dans resultats comme evaluer_ml, famille
#   LLM, scores par fold, prédictions par indice, conformité, troncature et coût.
# TODO : évaluer v0 et la version retenue sur les deux modèles, en groupant les appels par modèle.
# TODO : afficher les scores par fold, puis le F1 macro sur les 769 répliques et sur les 719 hors
#   jeu de mise au point.

> *Mes observations :*
>
> *Ma conclusion :*


## 4. Évaluation

### 4.1 Comparer sur les mêmes 769 répliques

Je rassemble F1 macro et accuracy par fold, moyenne et écart-type, pour la classe majoritaire, LinearSVC et chaque couple version et modèle. La dispersion décrit cinq scores et ne constitue pas un test de significativité.

Je compare sur les mêmes répliques, par la différence de F1 macro fold par fold avec LinearSVC, puis par les comptes de répliques où un modèle a raison et l'autre tort. Je lis les métriques par classe, car le F1 macro cache une classe abandonnée, et je commente chaque matrice de confusion.


In [18]:
# TODO : afficher scores par fold, moyennes et écarts-types des repères et de chaque couple version
#   et modèle, avec une colonne Modèle.
# TODO : calculer les différences par fold avec LinearSVC et les comptes « A juste, B faux ».
# TODO : afficher précision, rappel, F1 et supports par classe pour LinearSVC et la meilleure
#   version de chaque modèle.
# TODO : afficher les matrices de confusion sur les 769 prédictions, les sorties hors liste à part.

> *Mes observations :*
>
> *Ma conclusion :*


### 4.2 Revenir aux répliques

Je relis les erreurs réplique par réplique avec la sortie brute du modèle, car une étiquette fausse et une sortie hors liste ne racontent pas la même chose. Je suis JULIET–ROMEO et GLOUCESTER–DUKE VINCENTIO sans supposer que les erreurs du TP1 se reproduisent. Puis je sépare erreurs intra-pièce et inter-pièce avec le regroupement par voisinage prévu au TP2 (section 4.2).

Les deux modèles échouent-ils sur les mêmes répliques ? Je mesure leur taux d'accord et je lis les répliques qu'un seul des deux rattrape, car un accord élevé dirait que la limite vient de la tâche et non du modèle. Quelles erreurs le LLM corrige-t-il face à LinearSVC et lesquelles introduit-il ? Sur-prédit-il les personnages célèbres, ROMEO et JULIET, comme un a priori appris avant ce corpus ? Les exemples du prompt ont-ils copié un biais vers les noms cités ?


In [19]:
# TODO : afficher des répliques entières avec le locuteur réel, la prédiction LinearSVC, la sortie
#   brute et l'étiquette normalisée de chaque modèle.
# TODO : mesurer le taux d'accord entre les deux modèles, puis lister les répliques qu'un seul
#   rattrape.
# TODO : compter JULIET–ROMEO et GLOUCESTER–DUKE VINCENTIO par modèle, puis intra-pièce et
#   inter-pièce.
# TODO : lister les erreurs corrigées et introduites par le LLM face à LinearSVC, avec ou sans nom
#   cité.
# TODO : compter les prédictions par classe face aux effectifs réels, pour lire un a priori du
#   modèle.
# TODO : relire les sorties hors liste et les répliques que toutes les versions ratent.

> *Mes observations :*
>
> *Ma conclusion :*


### 4.3 Qualité, coût et latence

> *Mon hypothèse :* le LLM coûte au moins cent fois la latence de LinearSVC par réplique, et le grand modèle au moins deux fois celle du petit. Ce sont les tests de H5 et de H6.

Je compare les pipelines sur trois axes, la qualité en F1 macro, le coût en tokens et en temps, la latence par réplique. Les modèles sont locaux, donc le coût se lit en tokens, en secondes et en mémoire, et un prix par token le convertirait pour une API. Je ne garde que les mesures des appels réels stockés dans le cache, chargement exclu.

| Colonne | Ce qu'elle mesure |
|---|---|
| F1 macro, écart-type | Qualité moyenne sur les cinq folds et sa dispersion. |
| Conformité, troncature | Part des sorties lues comme une étiquette, et part des sorties coupées par la borne. |
| Tokens du prompt par réplique | Taille du contexte payé par appel, préfixe compris. |
| Tokens de réponse par réplique | Longueur de l'étiquette produite. |
| Tokens de raisonnement par réplique | Longueur du raisonnement interne, nulle en mode direct. |
| Latence par réplique (s) | Durée d'un appel hors chargement, ou temps de prédiction par réplique pour le ML. |
| Temps pour 769 répliques (s) | Coût d'une évaluation complète. |
| Empreinte mémoire (Gio) | Place du modèle sur la machine, zéro pour le ML. |
| Coût de bascule (s) | Chargement payé en changeant de modèle. |
| Entraînement (s) | Temps d'ajustement pour le ML, zéro pour le LLM. |

Le rapport de latence face à LinearSVC dépend de la configuration, donc je le donne par ligne et non comme un chiffre unique. Un pipeline hybride, LinearSVC d'abord puis le LLM sur les seuls cas incertains, est une piste si le LLM gagne en qualité mais perd en latence.


In [20]:
# TODO : construire le tableau qualité, coût et latence depuis resultats, une ligne par couple
#   version et modèle.
# TODO : mesurer le temps de prédiction par réplique de LinearSVC sur une validation de fold.
# TODO : ajouter l'empreinte mémoire et le coût de bascule mesurés dans les sections 2.2 et 2.5.
# TODO : afficher le tableau et le rapport de latence de chaque ligne face à LinearSVC.

> *Mes observations :*
>
> *Ma conclusion :*


### 4.4 Figer le prompt et évaluer sur le test

Je fige une version par modèle, sur le F1 macro moyen des cinq folds, sa dispersion, sa conformité et son coût. Ses exemples sont tirés une fois parmi les 769 répliques de développement avec la seed 42, et LinearSVC est réajusté sur ces 769 répliques.

Je compare les choix figés sur les 193 répliques de test, pour les deux modèles, avec le repère du TP1 de 0,563 pour LinearSVC. Le test sert ainsi à départager les modèles autant que les prompts. Puis j'analyse les erreurs sans relancer d'itération.

Ce test a déjà servi au TP1 pour choisir LinearSVC, donc la comparaison reste exploratoire.


In [21]:
# TODO : figer une VERSION_FINALE par modèle et ses exemples tirés parmi les 769 répliques de
#   développement.
# TODO : réajuster LinearSVC sur les 769 répliques, puis évaluer les trois sur les 193 répliques de
#   test.
# TODO : afficher scores, matrices de confusion, erreurs et coût dans un tableau final.

> *Mes observations :*
>
> *Ma conclusion :*


### 4.5 Mon journal d'expériences

Une ligne après chaque essai, et une hypothèse reste même si le résultat ne la confirme pas. Le registre affiché dans la section 3.4 fournit les chiffres.

| Expérience | Hypothèse | Changement | Observation chiffrée | Décision |
|---|---|---|---|---|
| 1. Déterminisme (section 2.4) | Deux appels identiques donnent le même texte | température 0 puis 0,7, seed 42, second appel hors cache | | |
| 2. Cache de préfixe (section 2.4) | Le second appel d'un prompt à trente exemples évalue moins de tokens | même prompt envoyé deux fois, puis une autre réplique | | |
| 3. Coût de bascule (section 2.5) | Alterner les modèles à chaque appel paierait un chargement par réplique | appels alternés contre appels groupés par modèle | | |
| 4. Modèle de mise au point (section 2.5) | Le modèle le plus rapide peut mener les itérations sans changer les décisions | latence par appel des deux modèles | | |
| 5. Mode de sortie (section 3.3) | H4, H8 | bloc `format`, v0 contre v1-texte, v1-json et v1-schema, sur les deux modèles | | |
| 6. v2 critères (section 3.4) | Des indices explicites corrigent les erreurs à nom cité | bloc `criteres` | | |
| 7. v3 contexte des pièces (section 3.4) | H2 | bloc `contexte` | | |
| 8. v4 un exemple par classe (section 3.4) | H3 | bloc `exemples`, dix exemples | | |
| 9. v5 trois exemples par classe (section 3.4) | H3, H5 | bloc `exemples`, trente exemples | | |
| 10. v6 exemples difficiles (section 3.4) | Les cas que LinearSVC rate guident mieux | choix des exemples | | |
| 11. Transfert du prompt (section 3.4) | H7 | mêmes versions rejouées sur le modèle de comparaison | | |
| 12. v7 raisonnement (section 3.5) | H8 | direct, raisonnement demandé, réflexion native | | |
| 13. Versions retenues sur les cinq folds (section 3.6) | H1, H3, H6 | jeu de mise au point remplacé par les folds, deux modèles | | |
| 14. Test final (section 4.4) | H1, H3, H6 | prompts figés, exemples tirés des 769, deux modèles | | |

### 4.6 Ce que je peux conclure

Je réponds à cinq questions. Que sait déjà le modèle de ces dix personnages, et que lui apportent le contexte et les exemples ? Quelle brique répond le mieux au problème, LinearSVC, un LLM ou les deux, scores, coût et latence compris ? Quel modèle pour quel budget, et le grand modèle justifie-t-il sa latence ? Quelles hypothèses, de H1 à H8, mes expériences soutiennent-elles ? Qu'est-ce qui reste incertain sur la conformité des sorties, le choix des exemples et la généralisation ?

> *Ma conclusion :*

### Pour aller plus loin

Prochaines pistes, hors de ce TP. Un ajustement fin d'un des deux modèles sur les 769 répliques, si le prompt plafonne. Une recherche des répliques voisines dans le train pour choisir les exemples de chaque appel, au prix du cache de préfixe. Un pipeline hybride où LinearSVC répond et le LLM n'arbitre que les cas incertains, pour la latence. Un modèle dérivé construit localement pour contrôler le raisonnement, si le bloc `format` ne suffit pas à le régler. Les log-probabilités du premier token généré, que le client peut demander, pour poser un seuil de confiance par classe comme au TP1. Des outils ou un agent seulement si une tâche l'exige, car je choisis la brique la plus simple qui suffit.
